# AOT Analyzer — ML Model Training
## Gold XAU/USD 15M Direction Predictor

### Instructions
1. Upload `xauusd_7months.csv` (raw OHLCV) to Colab
2. Run all cells below
3. Download `model_weights-2.json` → share with the AOT team

The notebook computes 28 technical features from raw OHLCV data,
trains a 64->32->16->1 neural net, and exports weights as JSON.

In [ ]:
import numpy as np
import pandas as pd
import json
import io
import os
import warnings
import math
from datetime import datetime, timezone
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix)
from sklearn.utils import class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)

In [ ]:
# Load raw OHLCV CSV
csv_filename = 'xauusd_7months.csv'

if os.path.exists(csv_filename):
    print(f'Found existing file: {csv_filename}')
    df = pd.read_csv(csv_filename, parse_dates=['time'])
else:
    print(f'Opening upload dialog...')
    from google.colab import files
    uploaded = files.upload()
    fn = list(uploaded.keys())[0]
    raw = uploaded[fn]
    if isinstance(raw, bytes):
        df = pd.read_csv(io.BytesIO(raw), parse_dates=['time'])
    else:
        df = pd.read_csv(io.StringIO(raw), parse_dates=['time'])
    print(f'Loaded via upload: {fn}')

df = df.sort_values('time').reset_index(drop=True)
print(f'Shape: {df.shape}')
print(f'Date range: {df["time"].iloc[0]} to {df["time"].iloc[-1]}')
print(f'Columns: {list(df.columns)}')

In [ ]:
# ---- Feature Engineering ----
# Compute all 28 features from raw OHLCV

opens = df['open'].values.astype(float)
highs = df['high'].values.astype(float)
lows = df['low'].values.astype(float)
closes = df['close'].values.astype(float)
volumes = df['volume'].values.astype(float)
n = len(df)

# Helper: SMA
def sma(arr, period):
    out = np.full(len(arr), np.nan)
    for i in range(period - 1, len(arr)):
        out[i] = np.mean(arr[i - period + 1:i + 1])
    return out

# Helper: EMA (Wilder-style, SMA seeded)
def ema(arr, period):
    k = 2.0 / (period + 1)
    out = np.full(len(arr), np.nan)
    if len(arr) < period:
        return out
    out[period - 1] = np.mean(arr[:period])
    for i in range(period, len(arr)):
        out[i] = (arr[i] - out[i - 1]) * k + out[i - 1]
    return out

# Helper: ATR (Wilder-smoothed)
def calc_atr(high, low, close, period=14):
    n = len(high)
    tr = np.zeros(n)
    for i in range(1, n):
        hl = high[i] - low[i]
        hc = abs(high[i] - close[i - 1])
        lc = abs(low[i] - close[i - 1])
        tr[i] = max(hl, hc, lc)
    out = np.full(n, np.nan)
    if n < period:
        return out
    out[period] = np.mean(tr[1:period + 1])
    alpha = 1.0 / period
    for i in range(period + 1, n):
        out[i] = out[i - 1] + alpha * (tr[i] - out[i - 1])
    return out

# Helper: RSI (Wilder-smoothed)
def calc_rsi(closes, period=14):
    n = len(closes)
    if n < period + 1:
        return np.full(n, 50.0)
    diffs = np.diff(closes)
    gains = np.where(diffs > 0, diffs, 0.0)
    losses = np.where(diffs < 0, -diffs, 0.0)
    avg_gain = np.mean(gains[:period])
    avg_loss = np.mean(losses[:period])
    out = np.full(n, 50.0)
    alpha = 1.0 / period
    for i in range(period, n - 1):
        avg_gain = avg_gain + alpha * (gains[i] - avg_gain)
        avg_loss = avg_loss + alpha * (losses[i] - avg_loss)
        rs = avg_gain / avg_loss if avg_loss > 0 else float('inf')
        rsi_val = 100.0 - 100.0 / (1.0 + rs) if math.isfinite(rs) else 100.0
        out[i + 1] = rsi_val
    return out

# Helper: Bollinger Bands (population std)
def calc_bb(closes, period=20, std_mult=2.0):
    n = len(closes)
    upper = np.full(n, np.nan)
    middle = np.full(n, np.nan)
    lower = np.full(n, np.nan)
    for i in range(period - 1, n):
        window = closes[i - period + 1:i + 1]
        m = np.mean(window)
        s = np.std(window)  # population std (ddof=0)
        upper[i] = m + std_mult * s
        middle[i] = m
        lower[i] = m - std_mult * s
    return upper, middle, lower

# Helper: ADX (Wilder-smoothed)
def calc_adx(high, low, close, period=14):
    n = len(high)
    if n < period * 2 + 1:
        return np.full(n, 25.0)
    tr = np.zeros(n)
    plus_dm = np.zeros(n)
    minus_dm = np.zeros(n)
    for i in range(1, n):
        hl = high[i] - low[i]
        hc = abs(high[i] - close[i - 1])
        lc = abs(low[i] - close[i - 1])
        tr[i] = max(hl, hc, lc)
        up_move = high[i] - high[i - 1]
        down_move = low[i - 1] - low[i]
        plus_dm[i] = up_move if up_move > down_move and up_move > 0 else 0
        minus_dm[i] = down_move if down_move > up_move and down_move > 0 else 0
    tr_smooth = np.sum(tr[1:period + 1])
    plus_smooth = np.sum(plus_dm[1:period + 1])
    minus_smooth = np.sum(minus_dm[1:period + 1])
    dx = np.full(n, np.nan)
    alpha = 1.0 / period
    for i in range(period + 1, n):
        tr_smooth += alpha * (tr[i] - tr_smooth)
        plus_smooth += alpha * (plus_dm[i] - plus_smooth)
        minus_smooth += alpha * (minus_dm[i] - minus_smooth)
        pdi = 100.0 * plus_smooth / tr_smooth if tr_smooth > 0 else 0
        ndi = 100.0 * minus_smooth / tr_smooth if tr_smooth > 0 else 0
        denom = pdi + ndi
        dx[i] = 100.0 * abs(pdi - ndi) / denom if denom > 0 else 0
    adx = np.full(n, 25.0)
    start = period + period
    if start >= n:
        return adx
    adx[start] = np.mean(dx[period + 1:start + 1])
    for i in range(start + 1, n):
        adx[i] = adx[i - 1] + alpha * (dx[i] - adx[i - 1])
    return adx

# ---- Compute all features ----
print('Computing features...')

df['atr'] = calc_atr(highs, lows, closes, 14)
df['ema20'] = ema(closes, 20)
df['ema50'] = ema(closes, 50)
df['ema200'] = ema(closes, 200)
df['sma50'] = sma(closes, 50)
df['rsi'] = calc_rsi(closes, 14)

bb_upper, bb_middle, bb_lower = calc_bb(closes, 20, 2.0)
df['bb_upper'] = bb_upper
df['bb_middle'] = bb_middle
df['bb_lower'] = bb_lower
df['bb_width'] = np.where(df['bb_middle'] > 0,
    (df['bb_upper'] - df['bb_lower']) / df['bb_middle'], 0.0)
df['bb_pct'] = np.where(df['bb_upper'] > df['bb_lower'],
    (closes - df['bb_lower']) / (df['bb_upper'] - df['bb_lower']), 0.5)

df['adx'] = calc_adx(highs, lows, closes, 14)

atr = df['atr']
df['price_dist_ema20_atr'] = np.where(atr > 0, (closes - df['ema20']) / atr, 0.0)
df['price_dist_ema50_atr'] = np.where(atr > 0, (closes - df['ema50']) / atr, 0.0)
df['ema20_dist_ema50_atr'] = np.where(atr > 0, (df['ema20'] - df['ema50']) / atr, 0.0)

# Range features (lookback excludes current candle)
def rolling_range_prev(high, low, period):
    out = np.full(len(high), np.nan)
    for i in range(period, len(high)):
        out[i] = np.max(high[i - period:i]) - np.min(low[i - period:i])
    return out

df['range_5'] = rolling_range_prev(highs, lows, 5)
df['range_20'] = rolling_range_prev(highs, lows, 20)
df['range_5_atr'] = np.where(atr > 0, df['range_5'] / atr, 0.0)
df['range_20_atr'] = np.where(atr > 0, df['range_20'] / atr, 0.0)

# Candle body / wick percentages
hl_range = highs - lows
candle_body = np.abs(closes - opens)
upper_wick = highs - np.maximum(closes, opens)
lower_wick = np.minimum(closes, opens) - lows
df['body_pct'] = np.where(hl_range > 0, candle_body / hl_range, 0.5)
df['upper_wick_pct'] = np.where(hl_range > 0, upper_wick / hl_range, 0.0)
df['lower_wick_pct'] = np.where(hl_range > 0, lower_wick / hl_range, 0.0)

# Volume change over 4 bars (misnamed vol3_change)
vol3 = np.full(n, np.nan)
for i in range(3, n):
    vol3[i] = (closes[i] - closes[i - 3]) / (closes[i - 3] or 1.0)
df['vol3_change'] = vol3

# Time-based features (from candle timestamp)
ts = df['time'].apply(lambda x: x.to_pydatetime() if hasattr(x, 'to_pydatetime') else x)
df['hour'] = ts.apply(lambda x: x.hour if hasattr(x, 'hour') else 0)
df['day_of_week'] = ts.apply(lambda x: x.weekday() if hasattr(x, 'weekday') else 0)
df['london_session'] = df['hour'].apply(lambda h: 1 if 8 <= h < 17 else 0)
df['ny_session'] = df['hour'].apply(lambda h: 1 if 13 <= h < 21 else 0)
df['asian_session'] = df['hour'].apply(lambda h: 1 if 0 <= h < 9 else 0)

# Target: 1 if next close > current close
df['target'] = (closes[1:] > closes[:-1]).astype(int).tolist() + [0]

# Drop rows with NaN (lookback window not yet filled)
before = len(df)
df = df.dropna().reset_index(drop=True)
print(f'Dropped {before - len(df)} rows with NaN (lookback). Remaining: {len(df)}')

In [ ]:
# Define feature columns (all except raw OHLCV, time, target)
drop_cols = ['time', 'open', 'high', 'low', 'close', 'volume', 'target']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df['target'].values

# Fill any remaining NaN with median
X = X.fillna(X.median())

print(f'Features: {len(feature_cols)}')
print(f'Samples: {len(X)}')
print(f'Target balance: UP={y.sum()}, DOWN={len(y)-y.sum()} ({y.mean()*100:.1f}% UP)')

In [ ]:
# Train/val/test split (chronological)
n = len(X)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
y_train = y[:train_end]
X_val = X.iloc[train_end:val_end]
y_val = y[train_end:val_end]
X_test = X.iloc[val_end:]
y_test = y[val_end:]

print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train.values)
X_val_s = scaler.transform(X_val.values)
X_test_s = scaler.transform(X_test.values)

# Class weights for imbalance
class_weights = class_weight.compute_class_weight(
    'balanced', classes=np.array([0, 1]), y=y_train
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f'Class weights: {class_weight_dict}')

In [ ]:
# Build model (64->32->16->1)
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_s.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)

model.summary()

In [ ]:
# Early stopping
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=100,
    batch_size=256,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Evaluate — find optimal threshold for highest win rate (precision)
y_prob = model.predict(X_test_s).flatten()

thresholds = np.arange(0.5, 0.99, 0.01)
best_threshold = 0.5
best_wr = 0

for thresh in thresholds:
    y_pred = (y_prob >= thresh).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        wr = tp / (tp + fp) if (tp + fp) > 0 else 0
        if wr > best_wr and (tp + fp) >= 20:
            best_wr = wr
            best_threshold = thresh

print(f'Best threshold for WR: {best_threshold:.2f}')
print(f'Win rate (precision) at threshold: {best_wr*100:.1f}%')
print(f'Signals at threshold: {int((y_prob >= best_threshold).sum())}')

# Final confusion matrix
y_pred_final = (y_prob >= best_threshold).astype(int)
cm = confusion_matrix(y_test, y_pred_final)
if cm.shape == (2, 2):
    tn, fp, fn, tp = cm.ravel()
    total_pred = tp + fp
    print(f'\nTrue UP: {tp}, False UP (losses): {fp}')
    print(f'WIN RATE: {tp/(tp+fp)*100:.1f}%' if total_pred > 0 else 'No UP predictions')

In [ ]:
# Export model weights for TypeScript/Python inference
def extract_weights(tf_model):
    layers_out = []
    for layer in tf_model.layers:
        if isinstance(layer, layers.Dense):
            w, b = layer.get_weights()
            activation = layer.activation.__name__
            layers_out.append({
                'weights': w.tolist(),
                'bias': b.tolist(),
                'activation': 'relu' if 'relu' in activation else 'sigmoid'
            })
    return layers_out

y_prob_all = model.predict(X_test_s).flatten()
y_pred_all = (y_prob_all >= best_threshold).astype(int)

acc = accuracy_score(y_test, y_pred_all)
prec = precision_score(y_test, y_pred_all, zero_division=0)
rec = recall_score(y_test, y_pred_all, zero_division=0)

model_json = {
    'feature_names': feature_cols,
    'mean': scaler.mean_.tolist(),
    'std': np.sqrt(scaler.var_).tolist(),
    'layers': extract_weights(model),
    'metadata': {
        'version': '2.0.0',
        'model_id': 'model-2',
        'trained_on': str(pd.Timestamp.now()),
        'accuracy': float(acc),
        'precision': float(prec),
        'recall': float(rec),
        'win_rate_at_threshold': float(best_wr),
        'decision_threshold': float(best_threshold),
        'data_range': f"{df['time'].iloc[0]} to {df['time'].iloc[-1]}",
        'samples': len(df),
        'features': len(feature_cols),
    }
}

with open('model_weights-2.json', 'w') as f:
    json.dump(model_json, f, indent=2)

print(f'Model weights exported to model_weights-2.json')
print(f'\nModel metadata:')
for k, v in model_json['metadata'].items():
    print(f'  {k}: {v}')

In [ ]:
# Download the model weights
from google.colab import files
files.download('model_weights-2.json')

### Done!
Share the downloaded `model_weights-2.json` with the AOT team.
The team will place it in both `bot/model_weights-2.json` and
`web/public/ml/model_weights-2.json`, then configure ensemble inference
that averages both this model and the original `model_weights.json`.

After integration, backtests will be run to compare the new win rate
against the current baseline (60.3% WR / 2.28 PF).